# Data Preprocessing and Normalization

For each dataset, a standard data preparation pipeline is applied in this notebook:

1. **Data Import**  
   Loading data from `.csv` or `.xlsx` files.

2. **Initial Inspection**  
   Examining the structure and data types using `.head()` and `.info()`.

3. **Transformation to Long Format**  
   Using `pd.melt()` to reshape the dataset into a format suitable for analysis (if required).

4. **Merging Excel Sheets**  
   For multi-sheet Excel files, data is combined using `reduce()` and `pd.merge()`.

5. **Saving Unique Region Names**  
   At the end of each step, a `.txt` file with unique region names is saved.  
   This is used later for consistent region naming and merging across datasets.


In [41]:
# import libraries
import pandas as pd

# data 1

In [42]:
# Load data
df_poverty = pd.read_csv("../../data/raw/poverty_percent_by_regions_1992_2020.csv")

# Preview first rows
display(df_poverty.head())

# Display dataframe info
print(df_poverty.info())


,region,year,poverty_percent
0,Российская Федерация,1992,33.5
1,Российская Федерация,1993,31.3
2,Российская Федерация,1994,22.4
3,Российская Федерация,1995,24.8
4,Российская Федерация,1996,22.1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2339 entries, 0 to 2338
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   region           2339 non-null   object 
 1   year             2339 non-null   int64  
 2   poverty_percent  2339 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 54.9+ KB
None


In [43]:
# Extract unique regions
unique_regions = df_poverty["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/poverty_unique_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 2

In [44]:
# для применения функции к элементам последовательности по очереди
from functools import reduce

# Загрузка данных
file_path = "../../data/raw/cash_real_income_wages_2015_2020.xlsx"

# Функция для обработки каждого листа
def process_income_sheet(sheet_name: str, value_column: str) -> pd.DataFrame:
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    # Удаляем строки, где все значения по годам — пропущены (например, округа)
    df = df.dropna(subset=df.columns[1:], how='all')
    # Переводим в long-формат
    df_long = pd.melt(df, id_vars=["region"], var_name="year", value_name=value_column)
    df_long["year"] = df_long["year"].astype(int)
    return df_long

# Обработка всех 4 листов
df_cash_income = process_income_sheet("per_capita_cash_income", "income_per_capita")
df_real_income = process_income_sheet("real_incomes", "real_income")
df_nominal_wage = process_income_sheet("formal_wage_paid", "nominal_wage")
df_real_wage = process_income_sheet("real_pay", "real_wage")

# Объединение по region и year
dfs = [df_cash_income, df_real_income, df_nominal_wage, df_real_wage]
df_merged_income = reduce(lambda left, right: pd.merge(left, right, on=["region", "year"], how="outer"), dfs)

# Посмотрим первые строки
display(df_merged_income.head())

# Посмотрим информацию о датафрейме
print(df_merged_income.info())

,region,year,income_per_capita,real_income,nominal_wage,real_wage
0,Алтайский край,2015,20860.0,99.1,20090.0,90
1,Алтайский край,2016,21256.0,94.7,21202.0,98.4
2,Алтайский край,2017,22139.0,100.0,22743.0,103.6
3,Алтайский край,2018,22829.0,99.7,25519.0,109.3
4,Алтайский край,2019,23937.0,99.6,27962.0,104.9


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   region             600 non-null    object 
 1   year               600 non-null    int64  
 2   income_per_capita  576 non-null    float64
 3   real_income        574 non-null    float64
 4   nominal_wage       576 non-null    float64
 5   real_wage          576 non-null    object 
dtypes: float64(3), int64(1), object(2)
memory usage: 28.3+ KB
None


In [45]:
# Extract unique regions
unique_regions = df_merged_income["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/cash_income_unique_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 3

In [46]:
# Load data
df_workers = pd.read_csv('../../data/raw/workers.csv', sep='\t')

# Display dataframe
df_workers

# Note: dataset contains no valid data

,11242000300190200001 Отношение числа занятых в экономике региона к численности населения региона в трудоспособном возрасте;;;;;;;;;;
0,ЕДИНИЦА ИЗМЕРЕНИЯ;Процент;;;;;;;;;
1,ТИП СБОРА;За период;;;;;;;;;
2,"ПЕРИОД;2014 г., 2012 г., 2015 г., 2016 г., 201..."
3,"ОКАТО;643 Российская Федерация, 030 Центральны..."


# data 4

In [47]:
# Load data
df_population = pd.read_excel("../../data/raw/population.xlsx", sheet_name="Отчет")

# Remove column with region codes
df_population = df_population.drop(columns=df_population.columns[1])

# Remove first 2 rows (metadata)
df_population = df_population.iloc[2:].reset_index(drop=True)

# Rename columns and keep years 2015–2020
df_population.columns = ["region_raw"] + [f"{2014 + i}" for i in range(len(df_population.columns) - 1)]
df_population = df_population[["region_raw", "2015", "2016", "2017", "2018", "2019", "2020"]]

# Remove row with month labels
df_population = df_population.iloc[1:].reset_index(drop=True)

# Convert to long format
df_population_clean = df_population.melt(id_vars="region_raw", var_name="year", value_name="population")
df_population_clean["year"] = df_population_clean["year"].astype(int)
df_population_clean["population"] = pd.to_numeric(df_population_clean["population"], errors="coerce")

# Preview result
df_population_clean.head()

# Prepare list for cleaned rows
clean_rows = []

# Iterate through rows in pairs (region name + value)
for i in range(len(df_population_clean) - 1):
    region = df_population_clean.iloc[i]["region_raw"]
    population = df_population_clean.iloc[i + 1]["population"]
    year = df_population_clean.iloc[i + 1]["year"]

    # If current row contains region name (NaN in population)
    # and next row contains actual value
    if pd.isna(df_population_clean.iloc[i]["population"]) and pd.notna(population):
        clean_rows.append({"region": region, "year": year, "population": population})

# Create final dataframe
df_population_fixed = pd.DataFrame(clean_rows)

# Preview final result
display(df_population_fixed.head())

# Display dataframe info
print(df_population_fixed.info())

,region,year,population
0,Центральный федеральный округ,2015,38227656.0
1,Белгородская область,2015,1501699.0
2,Брянская область,2015,1423178.0
3,Владимирская область,2015,1575507.0
4,Воронежская область,2015,2441337.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568 entries, 0 to 567
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   region      568 non-null    object 
 1   year        568 non-null    int64  
 2   population  568 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 13.4+ KB
None


In [48]:
# Extract unique regions
unique_regions = df_population_fixed["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/population_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 5

In [49]:
# Load data
file_path = "../../data/raw/gross_regional_product_1996_2020.xls"

# Load Excel file using xlrd engine
xls = pd.ExcelFile(file_path, engine="xlrd")

# Read first sheet (no headers)
df = pd.read_excel(xls, sheet_name=0, header=None)

# Set third row as column names
df.columns = df.iloc[2]

# Remove first two rows (metadata)
df = df.drop(index=[0, 1]).reset_index(drop=True)

# Rename first column (region names)
df = df.rename(columns={df.columns[0]: "region"})

# Convert to long format
df_gross = df.melt(id_vars="region", var_name="year", value_name="grp_per_capita")

# Remove rows with missing values
df_gross = df_gross.dropna(subset=["region", "grp_per_capita"])

# Remove aggregated regions (federal districts)
df_gross = df_gross[~df_gross["region"].str.contains("федеральный округ", case=False)]

# Convert data types
df_gross["year"] = df_gross["year"].astype(int)
df_gross["grp_per_capita"] = df_gross["grp_per_capita"].astype(float)

# Preview data
display(df_gross.head())

# Display dataframe info
print(df_gross.info())

,region,year,grp_per_capita
1,Российская Федерация,1996,12225.0
3,Белгородская область,1996,9575.6
4,Брянская область,1996,7275.3
5,Владимирская область,1996,7620.7
6,Воронежская область,1996,7651.9


<class 'pandas.core.frame.DataFrame'>
Index: 2186 entries, 1 to 2548
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   region          2186 non-null   object 
 1   year            2186 non-null   int64  
 2   grp_per_capita  2186 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 68.3+ KB
None


In [50]:
# Extract unique regions
unique_regions = df_gross["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/gross_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 6 

In [51]:
# Load data
df = pd.read_excel('../../data/raw/retail_turnover_per_capita_2000_2021.xls', header=None)

# Use third row as column names
new_columns = df.iloc[2].tolist()
new_columns[0] = "region"
df.columns = new_columns

# Remove first three rows (metadata)
df = df.drop(index=[0, 1, 2]).reset_index(drop=True)

# Remove column with measurement units
df = df.drop(columns=df.columns[1])

# Convert to long format
df_retail = df.melt(id_vars="region", var_name="year", value_name="retail_per_capita")

# Clean numeric values
df_retail["retail_per_capita"] = (
    df_retail["retail_per_capita"]
    .astype(str)
    .str.replace("\xa0", "", regex=False)  # remove non-breaking spaces
    .str.replace(",", ".", regex=False)    # replace comma with dot
)

# Remove non-numeric rows
df_retail = df_retail[df_retail["retail_per_capita"].str.match(r"^\d+(\.\d+)?$")]

# Convert data types
df_retail["retail_per_capita"] = df_retail["retail_per_capita"].astype(float)
df_retail["year"] = df_retail["year"].astype(int)

# Preview data
display(df_retail.head())

# Display dataframe info
print(df_retail.info())

,region,year,retail_per_capita
1,Российская Федерация,2000,16046.0
2,Центральный федеральный округ,2000,26062.0
3,Белгородская область,2000,11820.0
4,Брянская область,2000,8267.0
5,Владимирская область,2000,7442.0


<class 'pandas.core.frame.DataFrame'>
Index: 2142 entries, 1 to 2396
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   region             2142 non-null   object 
 1   year               2142 non-null   int64  
 2   retail_per_capita  2142 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 66.9+ KB
None


In [52]:
# Extract unique regions
unique_regions = df_retail["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/retail_turnover_per_capita_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 7

In [53]:
# Load data
df = pd.read_excel('../../data/raw/child_mortality_urban_1990_2021.xls', header=None)

# Use third row as column names
new_columns = df.iloc[2].tolist()
new_columns[0] = "region"
df.columns = new_columns

# Remove first three rows (metadata)
df = df.drop(index=[0, 1, 2]).reset_index(drop=True)

# Remove columns with measurement units
df = df.drop(columns=df.columns[1:3])

# Convert to long format
df_child_urban = df.melt(id_vars="region", var_name="year", value_name="infant_mortality")

# Clean numeric values
df_child_urban["infant_mortality"] = (
    df_child_urban["infant_mortality"]
    .astype(str)
    .str.replace("\xa0", "", regex=False)  # remove non-breaking spaces
    .str.replace(",", ".", regex=False)    # replace comma with dot
)

# Remove non-numeric rows
df_child_urban = df_child_urban[df_child_urban["infant_mortality"].str.match(r"^\d+(\.\d+)?$")]

# Convert data types
df_child_urban["infant_mortality"] = df_child_urban["infant_mortality"].astype(float)
df_child_urban["year"] = df_child_urban["year"].astype(int)

# Preview data
display(df_child_urban.head())

# Display dataframe info
print(df_child_urban.info())

,region,year,infant_mortality
0,Российская Федерация,1990,23902.0
1,Центральный федеральный округ,1990,5317.0
2,Белгородская область,1990,209.0
3,Брянская область,1990,198.0
4,Владимирская область,1990,221.0


<class 'pandas.core.frame.DataFrame'>
Index: 3176 entries, 0 to 3794
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   region            3176 non-null   object 
 1   year              3176 non-null   int64  
 2   infant_mortality  3176 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 99.2+ KB
None


In [54]:
# Extract unique regions
unique_regions = df_child_urban["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/child_mortality_urban_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 8

In [55]:
# Load data
df = pd.read_excel('../../data/raw/child_mortality_rural_1990_2021.xls', header=None)

# Use third row as column names
new_columns = df.iloc[2].tolist()
new_columns[0] = "region"
df.columns = new_columns

# Remove first three rows (metadata)
df = df.drop(index=[0, 1, 2]).reset_index(drop=True)

# Remove columns with measurement units
df = df.drop(columns=df.columns[1:3])

# Convert to long format
df_child_rural = df.melt(id_vars="region", var_name="year", value_name="infant_mortality")

# Clean numeric values
df_child_rural["infant_mortality"] = (
    df_child_rural["infant_mortality"]
    .astype(str)
    .str.replace("\xa0", "", regex=False)  # remove non-breaking spaces
    .str.replace(",", ".", regex=False)    # replace comma with dot
)

# Remove non-numeric rows
df_child_rural = df_child_rural[df_child_rural["infant_mortality"].str.match(r"^\d+(\.\d+)?$")]

# Convert data types
df_child_rural["infant_mortality"] = df_child_rural["infant_mortality"].astype(float)
df_child_rural["year"] = df_child_rural["year"].astype(int)

# Preview data
display(df_child_rural.head())

# Display dataframe info
print(df_child_rural.info())

,region,year,infant_mortality
0,Российская Федерация,1990,11186.0
1,Центральный федеральный округ,1990,1615.0
2,Белгородская область,1990,103.0
3,Брянская область,1990,124.0
4,Владимирская область,1990,80.0


<class 'pandas.core.frame.DataFrame'>
Index: 3159 entries, 0 to 3794
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   region            3159 non-null   object 
 1   year              3159 non-null   int64  
 2   infant_mortality  3159 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 98.7+ KB
None


In [56]:
# Extract unique regions
unique_regions = df_child_rural["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/child_mortality_rural_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 9

In [57]:
# Load data
df_disabled = pd.read_csv('../../data/raw/disabled_total_by_age_2017_2022.csv')

# Remove rows with missing values
df_disabled = df_disabled.dropna()

# Preview data
display(df_disabled.head())

# Display dataframe info
print(df_disabled.info())

,region,total,18_30,31_40,41_50,51_60,60_,date
0,Российская Федерация,11640873.0,550895.0,766054.0,1030652.0,2135436.0,7157836.0,2017-01-01
1,Центральный федеральный округ,3420310.0,118579.0,172662.0,257484.0,598102.0,2273483.0,2017-01-01
2,Белгородская область,223030.0,6318.0,10383.0,16596.0,37444.0,152289.0,2017-01-01
3,Брянская область,110418.0,4215.0,6568.0,10230.0,21481.0,67924.0,2017-01-01
4,Владимирская область,133352.0,4454.0,6811.0,9606.0,23322.0,89159.0,2017-01-01


<class 'pandas.core.frame.DataFrame'>
Index: 6073 entries, 0 to 6079
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   region  6073 non-null   object 
 1   total   6073 non-null   float64
 2   18_30   6073 non-null   float64
 3   31_40   6073 non-null   float64
 4   41_50   6073 non-null   float64
 5   51_60   6073 non-null   float64
 6   60_     6073 non-null   float64
 7   date    6073 non-null   object 
dtypes: float64(6), object(2)
memory usage: 427.0+ KB
None


In [58]:
# Extract unique regions
unique_regions = df_disabled["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/disabled_total_by_age_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 10

In [59]:
# Load data
df = pd.read_excel('../../data/raw/morbidity_2005_2020_age_disease.xls', header=None)

# Extract column names (years + first 3 columns)
new_columns = df.iloc[2].copy()
new_columns.iloc[0:3] = ["region", "disease", "age"]

# Create cleaned dataframe starting from row 3
df_clean = df.iloc[3:].copy()
df_clean.columns = new_columns

# Remove rows with missing region or disease
df_clean = df_clean.dropna(subset=["region", "disease"])

# Convert numeric values to float
for col in df_clean.columns[3:]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Remove rows where all numeric columns are missing
df_clean = df_clean.dropna(how='all', subset=df_clean.columns[3:])

# Convert to long format
df_morbidity = pd.melt(
    df_clean,
    id_vars=["region", "disease", "age"],
    var_name="year",
    value_name="cases"
)

# Convert year to integer
df_morbidity["year"] = df_morbidity["year"].astype(int)

# Remove rows with missing values in "cases"
df_morbidity.dropna(subset=["cases"], inplace=True)

# Preview data
display(df_morbidity.head())

# Display dataframe info
print(df_morbidity.info())

,region,disease,age,year,cases
0,Российская Федерация,"Беременность, роды и послеродовой период",0-14 лет,2005,21.3
1,Российская Федерация,"Беременность, роды и послеродовой период",15-17 лет,2005,1537.3
2,Российская Федерация,"Беременность, роды и послеродовой период",18 лет и старше,2005,6731.7
3,Российская Федерация,"Беременность, роды и послеродовой период",Всего,2005,5719.4
4,Российская Федерация,Болезни глаза и его придаточного аппарата,0-14 лет,2005,5643.4


<class 'pandas.core.frame.DataFrame'>
Index: 75451 entries, 0 to 97839
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   region   75451 non-null  object 
 1   disease  75451 non-null  object 
 2   age      75451 non-null  object 
 3   year     75451 non-null  int64  
 4   cases    75451 non-null  float64
dtypes: float64(1), int64(1), object(3)
memory usage: 3.5+ MB
None


In [60]:
# Extract unique regions
unique_regions = df_morbidity["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/morbidity_2005_2020_age_disease_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 11

In [61]:
# Load data
df_welfare = pd.read_excel("../../data/raw/welfare_expense_share_2015_2020.xlsx", sheet_name="data")

# Remove rows where all yearly values are missing (e.g., aggregated regions)
df_welfare_cleaned = df_welfare.dropna(
    subset=[2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0],
    how='all'
)

# Convert from wide to long format
df_welfare_long = pd.melt(
    df_welfare_cleaned,
    id_vars=["region"],
    var_name="year",
    value_name="welfare_percent"
)

# Convert year to integer
df_welfare_long["year"] = df_welfare_long["year"].astype(int)

# Preview data
display(df_welfare_long.head())

# Display dataframe info
print(df_welfare_long.info())

,region,year,welfare_percent
0,Российская Федерация,2015,15.8
1,Белгородская область,2015,11.3
2,Брянская область,2015,22.0
3,Владимирская область,2015,18.1
4,Воронежская область,2015,15.2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 516 entries, 0 to 515
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   region           516 non-null    object 
 1   year             516 non-null    int64  
 2   welfare_percent  516 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 12.2+ KB
None


In [62]:
# Extract unique regions
unique_regions = df_welfare_long["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/welfare_expense_share_2015_2020_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 12

In [63]:
# Load data
df1 = pd.read_csv("../../data/raw/regional_production_2005_2016.csv")
df2 = pd.read_csv("../../data/raw/regional_production_2017_2020.csv")

# Convert to long format
df1_long = pd.melt(df1, id_vars=["region", "production_field"],
                   var_name="year", value_name="value")
df2_long = pd.melt(df2, id_vars=["region", "production_field"],
                   var_name="year", value_name="value")

# Combine datasets
df_regional_production = pd.concat([df1_long, df2_long], ignore_index=True)

# Convert data types and clean missing values
df_regional_production["year"] = df_regional_production["year"].astype(int)
df_regional_production = df_regional_production.dropna(subset=["value"])

# Preview data
display(df_regional_production.head())

# Display dataframe info
print(df_regional_production.info())

,region,production_field,year,value
0,Российская Федерация,РАЗДЕЛ С ДОБЫЧА ПОЛЕЗНЫХ ИСКОПАЕМЫХ,2005,3.062460e+09
1,Российская Федерация,Подраздел СА ДОБЫЧА ТОПЛИВНО-ЭНЕРГЕТИЧЕСКИ...,2005,2.686256e+09
2,Российская Федерация,"Добыча каменного угля,бурого угля и торфа",2005,2.321794e+08
3,Российская Федерация,Добыча сырой нефти и природного газа; ...,2005,2.450541e+09
4,Российская Федерация,Добыча урановой и ториевой руд,2005,3.535251e+06


<class 'pandas.core.frame.DataFrame'>
Index: 11276 entries, 0 to 13563
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   region            11276 non-null  object 
 1   production_field  11276 non-null  object 
 2   year              11276 non-null  int64  
 3   value             11276 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 440.5+ KB
None


In [64]:
# Extract unique regions
unique_regions = df_regional_production["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/regional_production_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 13

In [65]:
# Search for files using pattern
import glob

# Load data (all matching files)
socdem_files = sorted(glob.glob("../../data/raw/poverty_socdem_20*.xls"))

# Cleaning function for each file
def clean_socdem(filepath):
    year = int(filepath.split("_")[-1].split(".")[0])
    
    df = pd.read_excel(filepath, header=None)
    df_clean = df.iloc[3:].copy()
    
    # Extract column names
    new_columns = df.iloc[2].tolist()
    new_columns[0] = "region"
    df_clean.columns = new_columns

    # Remove column with total values (100%)
    df_clean = df_clean.drop(columns=[new_columns[1]])

    # Add year column
    df_clean["year"] = year

    # Convert numeric columns
    for col in df_clean.columns[1:]:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    return df_clean.reset_index(drop=True)

# Process all files and combine into one dataset
df_socdem_all = pd.concat([clean_socdem(f) for f in socdem_files], ignore_index=True)

# Preview data
display(df_socdem_all.head())

# Display dataframe info
print(df_socdem_all.info())

,region,Дети в возрасте до 16 лет,Население старше трудоспособного возраста,Население трудоспособного возраста,year
0,Российская Федерация,39.3,6.6,54.1,2017
1,Белгородская область,43.4,11.8,44.8,2017
2,Брянская область,42.9,4.9,52.2,2017
3,Владимирская область,34.8,8.6,56.6,2017
4,Воронежская область,38.6,5.9,55.6,2017


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352 entries, 0 to 351
Data columns (total 5 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   region                                     352 non-null    object 
 1   Дети в возрасте до 16 лет                  340 non-null    float64
 2   Население старше трудоспособного возраста  340 non-null    float64
 3   Население трудоспособного возраста         340 non-null    float64
 4   year                                       352 non-null    int64  
dtypes: float64(3), int64(1), object(1)
memory usage: 13.9+ KB
None


In [66]:
# Extract unique regions
unique_regions = df_socdem_all["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/poverty_socdem_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 14

In [67]:
# Load data
file_path = "../../data/raw/housing_2020.xlsx"
df_cond = pd.read_excel(file_path, sheet_name="housing_cond")
df_intent = pd.read_excel(file_path, sheet_name="housing_intent")

# Remove rows where all values except region are missing
df_cond = df_cond.dropna(subset=df_cond.columns[1:], how="all")
df_intent = df_intent.dropna(subset=df_intent.columns[1:], how="all")

# Rename columns
df_cond = df_cond.rename(columns={
    df_cond.columns[0]: "region",
    df_cond.columns[1]: "all_households_pct",
    df_cond.columns[2]: "not_crowded_pct",
    df_cond.columns[3]: "somewhat_crowded_pct",
    df_cond.columns[4]: "very_crowded_pct",
    df_cond.columns[5]: "no_answer_pct",
    df_cond.columns[6]: "total_area_per_person",
    df_cond.columns[7]: "living_area_per_person",
    df_cond.columns[8]: "rooms_per_household"
})

df_intent = df_intent.rename(columns={
    df_intent.columns[0]: "region",
    df_intent.columns[1]: "all_households",
    df_intent.columns[2]: "intend_improve_housing",
    df_intent.columns[3]: "intend_due_to_crowding",
    df_intent.columns[4]: "intend_due_to_bad_condition",
    df_intent.columns[5]: "intend_due_to_both",
    df_intent.columns[6]: "plan_dolevka",
    df_intent.columns[7]: "plan_queue",
    df_intent.columns[8]: "plan_resettle",
    df_intent.columns[9]: "plan_buy_or_build",
    df_intent.columns[10]: "plan_rent",
    df_intent.columns[11]: "plan_other",
    df_intent.columns[12]: "no_answer",
    df_intent.columns[13]: "not_intending_to_improve"
})

# Remove baseline columns (always equal to 100%)
df_cond = df_cond.drop(columns=["all_households_pct"])
df_intent = df_intent.drop(columns=["all_households"])

# Merge datasets by region
df_housing = pd.merge(df_cond, df_intent, on="region", how="outer")

# Add year column
df_housing["year"] = 2020

# Preview data
display(df_housing.head())

# Display dataframe info
print(df_housing.info())

,region,not_crowded_pct,somewhat_crowded_pct,very_crowded_pct,no_answer_pct,total_area_per_person,living_area_per_person,rooms_per_household,intend_improve_housing,intend_due_to_crowding,...,intend_due_to_both,plan_dolevka,plan_queue,plan_resettle,plan_buy_or_build,plan_rent,plan_other,no_answer,not_intending_to_improve,year
0,Bладимирская область,81.8,15.8,2.5,0.0,24.2,16.1,2.2,9.0,3.8,...,0.0,13.8,1.7,0,38.4,1.5,44.5,0,91.0,2020
1,Bолгоградская область,82.7,13.9,3.3,0.0,25.6,17.9,2.5,12.1,6.2,...,0.6,2,3.3,0,39.5,3.2,52.4,0,87.4,2020
2,Bологодская область,87.5,9.6,2.9,0.0,22.9,16.9,2.2,13.6,3.4,...,0.3,6.1,2.3,2.9,43.5,2.7,42.5,0,86.4,2020
3,Bоронежская область,87.0,11.4,1.6,0.1,29.0,19.9,2.6,14.2,4.2,...,0.1,15.4,3.7,0.3,29.4,1.4,49.7,0,85.5,2020
4,Алтайский край,83.2,13.4,3.3,0.1,25.9,18.0,2.4,14.9,6.3,...,0.2,3.1,5,0.8,31.5,3.9,56.2,0.6,85.1,2020


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   region                       86 non-null     object 
 1   not_crowded_pct              86 non-null     float64
 2   somewhat_crowded_pct         86 non-null     float64
 3   very_crowded_pct             86 non-null     float64
 4   no_answer_pct                86 non-null     float64
 5   total_area_per_person        86 non-null     float64
 6   living_area_per_person       86 non-null     float64
 7   rooms_per_household          86 non-null     float64
 8   intend_improve_housing       86 non-null     float64
 9   intend_due_to_crowding       86 non-null     float64
 10  intend_due_to_bad_condition  86 non-null     float64
 11  intend_due_to_both           86 non-null     float64
 12  plan_dolevka                 86 non-null     object 
 13  plan_queue            

In [68]:
# Extract unique regions
unique_regions = df_housing["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/housing_regions.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 15

In [69]:
# Load data
file_path = "../../data/raw/drug_alco.xlsx"

# Function to process a single sheet
def process_drug_sheet(sheet_name: str, diagnosis: str) -> pd.DataFrame:
    # Read data
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    
    # Convert from wide to long format
    df_long = pd.melt(df, id_vars=[df.columns[0]], var_name="year", value_name="value")
    
    # Rename columns
    df_long.columns = ["region", "year", "value"]
    
    # Add diagnosis label
    df_long["diagnosis"] = diagnosis
    
    # Convert year to integer
    df_long["year"] = df_long["year"].astype(int)
    
    # Remove rows with missing values
    return df_long.dropna(subset=["value"])

# Process all sheets
alco_old = process_drug_sheet("alco", "alcohol")
alco_new = process_drug_sheet("alco1718", "alcohol")
drug_old = process_drug_sheet("drugs", "drugs")
drug_new = process_drug_sheet("drug1718", "drugs")

# Combine datasets
df_drug_alco = pd.concat([alco_old, alco_new, drug_old, drug_new], ignore_index=True)

# Preview data
display(df_drug_alco.head())

# Display dataframe info
print(df_drug_alco.info())

,region,year,value,diagnosis
0,Российская Федерация,2005,147.4,alcohol
1,Центральный федеральный округ,2005,141.5,alcohol
2,Белгородская область,2005,99.6,alcohol
3,Брянская область,2005,243.8,alcohol
4,Владимирская область,2005,178.3,alcohol


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2486 entries, 0 to 2485
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   region     2486 non-null   object
 1   year       2486 non-null   int64 
 2   value      2486 non-null   object
 3   diagnosis  2486 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB
None


In [70]:
# Extract unique regions
unique_regions = df_drug_alco["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/drug_alco.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

# data 16

In [71]:
# Load data
df = pd.read_csv("../../data/raw/newborn_2006_2022_monthly.csv", sep=";", encoding="utf-8")

# Remove unnecessary columns
df = df.loc[:, ~df.columns.str.contains("Unnamed")]

# Convert numeric values (comma → dot → float)
for col in df.columns[1:]:
    df[col] = df[col].astype(str).str.replace(",", ".", regex=False).astype(float)

# Convert from wide to long format
df_long = pd.melt(df, id_vars=["Region"], var_name="month_year", value_name="births")

# Map month names to numeric format
month_map = {
    "январь": "01", "февраль": "02", "март": "03", "апрель": "04",
    "май": "05", "июнь": "06", "июль": "07", "август": "08",
    "сентябрь": "09", "октябрь": "10", "ноябрь": "11", "декабрь": "12"
}

df_long["month"] = df_long["month_year"].str.extract(r"^(\w+)", expand=False).map(month_map)
df_long["year"] = df_long["month_year"].str.extract(r"(\d{4})").astype(int)

# Create datetime column from year and month
df_long["date"] = pd.to_datetime(
    df_long["year"].astype(str) + "-" + df_long["month"],
    errors="coerce"
)

# Rename column
df_long.rename(columns={"Region": "region"}, inplace=True)

# Aggregate data by year
df_yearly = df_long.dropna(subset=["births", "date"])
df_yearly["year"] = df_yearly["date"].dt.year
df_births_by_year = df_yearly.groupby(["region", "year"], as_index=False)["births"].sum()

# Preview data
display(df_births_by_year.head())

# Display dataframe info
print(df_births_by_year.info())

/var/folders/m5/5wwwb0bj5jddhlnkw6777l600000gn/T/ipykernel_6357/182943423.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_yearly["year"] = df_yearly["date"].dt.year


,region,year,births
0,Агинский Бурятский округ (Забайкальский край),2006,1335.0
1,Агинский Бурятский округ (Забайкальский край),2007,1531.0
2,Агинский Бурятский округ (Забайкальский край),2008,1756.0
3,Агинский Бурятский округ (Забайкальский край),2009,1738.0
4,Агинский Бурятский округ (Забайкальский край),2010,1831.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1633 entries, 0 to 1632
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   region  1633 non-null   object 
 1   year    1633 non-null   int32  
 2   births  1633 non-null   float64
dtypes: float64(1), int32(1), object(1)
memory usage: 32.0+ KB
None


In [72]:
# Extract unique regions
unique_regions = df_births_by_year["region"].unique()

# Save unique regions to a text file
with open("../../outputs/unique_regions_dict/newborn.txt", "w", encoding="utf-8") as f:
    for region in unique_regions:
        f.write(region + "\n")

## Region Name Unification and Dataset Standardization

The next step in preprocessing is to standardize region names across all datasets in order to ensure consistent merging and analysis.

This stage includes:
- collecting all unique region names from `.txt` files
- creating a `region_mapping` dictionary to normalize naming variations
- defining a list of reference regions (excluding `None` values and aggregated regions)
- implementing a function to standardize region names in dataframes
- applying this function to all datasets and saving the results as `*_standardized.csv`


In [73]:
# Get paths to all .txt files
files = glob.glob("../../outputs/unique_regions_dict/*.txt")

# List to store all region names
all_regions = []

# Iterate over all files
for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        
        # Remove newline characters and extra spaces
        cleaned = [line.strip() for line in lines if line.strip()]
        all_regions.extend(cleaned)

# Keep only unique values
unique_regions = sorted(set(all_regions))

# Display result
print("Total number of unique region names across all files:")
print(len(unique_regions))

Total number of unique region names across all files:
188


In [74]:
# The dictionary maps inconsistent or alternative region names
# to standardized (canonical) region names.
# Entries with None represent aggregated regions or invalid entries to be excluded.

region_mapping = {

    "Bладимирская область": "Владимирская область",
    "Владимирская область": "Владимирская область",
    "Bоронежская область": "Воронежская область",
    "Воронежская область": "Воронежская область",
    "Bологодская область": "Вологодская область",
    "Вологодская область": "Вологодская область",
    "Белгородская область": "Белгородская область",
    "Брянская область": "Брянская область",
    "Ивановская область": "Ивановская область",
    "Калужская область": "Калужская область",
    "Костромская область": "Костромская область",
    "Курская область": "Курская область",
    "Липецкая область": "Липецкая область",
    "Московская область": "Московская область",
    "Московская обл. в старых границах": "Московская область",
    "Орловская область": "Орловская область",
    "Рязанская область": "Рязанская область",
    "Смоленская область": "Смоленская область",
    "Тамбовская область": "Тамбовская область",
    "Тверская область": "Тверская область",
    "Тульская область": "Тульская область",
    "Ярославская область": "Ярославская область",
    

    "Архангельская область": "Архангельская область",
    "Архангельская область (без АО)": "Архангельская область",
    "Архангельская область (кроме Ненецкого автономного округа)": "Архангельская область",
    "Архангельская область без": "Архангельская область",
    "Архангельская область без авт. округа": "Архангельская область",
    "Архангельская область без автономного округа": "Архангельская область",
    "Архангельская обл. без данных по Ненецкому авт. окр.": "Архангельская область",
    "Вологодская область": "Вологодская область",
    "Калининградская область": "Калининградская область",
    "Ленинградская область": "Ленинградская область",
    "Мурманская область": "Мурманская область",
    "Новгородская область": "Новгородская область",
    "Псковская область": "Псковская область",
    "г. Санкт-Петербург": "Санкт-Петербург",
    "г. Севастополь": "Севастополь",
    "город Санкт-Петербург": "Санкт-Петербург",
    "город Севастополь": "Севастополь",
    

    "Астраханская область": "Астраханская область",
    "Волгоградская область": "Волгоградская область",
    "Ростовская область": "Ростовская область",
    "Республика Адыгея": "Республика Адыгея",
    "Республика Адыгея (Адыгея)": "Республика Адыгея",
    "Республика Калмыкия": "Республика Калмыкия",
    "Краснодарский край": "Краснодарский край",
    "Ставропольский край": "Ставропольский край",
    "Кабардино-Балкарская Республика": "Кабардино-Балкарская Республика",
    "Кабардино-Балкарская": "Кабардино-Балкарская Республика",
    "Карачаево-Черкесская Республика": "Карачаево-Черкесская Республика",
    "Карачаево-Черкесская": "Карачаево-Черкесская Республика",
    "Чеченская Республика": "Чеченская Республика",
    "Республика Ингушетия": "Республика Ингушетия",
    "Республика Северная Осетия - Алания": "Республика Северная Осетия - Алания",
    "Республика Северная Осетия-Алания": "Республика Северная Осетия - Алания",
    "Осетия-Алания": "Республика Северная Осетия - Алания",
    "Республика Дагестан": "Республика Дагестан",



    "Нижегородская область": "Нижегородская область",
    "Кировская область": "Кировская область",
    "Пензенская область": "Пензенская область",
    "Самарская область": "Самарская область",
    "Саратовская область": "Саратовская область",
    "Ульяновская область": "Ульяновская область",
    "Республика Мордовия": "Республика Мордовия",
    "Республика Марий Эл": "Республика Марий Эл",
    "Чувашская Республика": "Чувашская Республика",
    "Чувашская Республика - Чувашия": "Чувашская Республика",
    "Республика Татарстан": "Республика Татарстан",
    "Республика Татарстан (Татарстан)": "Республика Татарстан",
    "Республика Башкортостан": "Республика Башкортостан",
    "Удмуртская Республика": "Удмуртская Республика",
    

    "Свердловская область": "Свердловская область",
    "Челябинская область": "Челябинская область",
    "Пермский край": "Пермский край",
    "Курганская область": "Курганская область",
    "Тюменская область": "Тюменская область",
    "Тюменская обл.без данных по Ханты-Мансийскому и Ямало-Ненецкому авт. окр.": "Тюменская область",
    "Тюменская область (без АО)": "Тюменская область",
    "Тюменская область (кроме Ханты-Мансийского автономного округа-Югры и Ямало-Ненецкого автономного округа)": "Тюменская область",
    "Тюменская область без": "Тюменская область",
    "Тюменская область без авт. округов": "Тюменская область",
    "Тюменская область без автономного округа": "Тюменская область",
    "Ханты-Мансийский автономный округ": "Ханты-Мансийский автономный округ",
    "Ханты-Мансийский авт. округ": "Ханты-Мансийский автономный округ",
    "Ханты-Мансийский автономный округ - Югра (Тюменская область)": "Ханты-Мансийский автономный округ",
    "Ханты-Мансийский АО": "Ханты-Мансийский автономный округ",
    "Ханты-Мансийский": "Ханты-Мансийский автономный округ",
    "Ямало-Hенецкий АО": "Ямало-Ненецкий автономный округ",
    "Ямало-Ненецкий": "Ямало-Ненецкий автономный округ",
    "Ямало-Ненецкий авт. округ": "Ямало-Ненецкий автономный округ",
    "Ямало-Ненецкий автономный округ": "Ямало-Ненецкий автономный округ",
    "Ямало-Ненецкий автономный округ (Тюменская область)": "Ямало-Ненецкий автономный округ",
    

    "Алтайский край": "Алтайский край",
    "Красноярский край": "Красноярский край",
    "Кемеровская область": "Кемеровская область",
    "Кемеровская область - Кузбасс": "Кемеровская область",
    "Новосибирская область": "Новосибирская область",
    "Омская область": "Омская область",
    "Томская область": "Томская область",
    "Иркутская область": "Иркутская область",
    "Республика Алтай": "Республика Алтай",
    "Республика Хакасия": "Республика Хакасия",
    "Республика Тыва": "Республика Тыва",
    "Бурятия": "Республика Бурятия",
    "Республика Бурятия": "Республика Бурятия",
    "Республика Саха (Якутия)": "Республика Саха (Якутия)",
    "Республика Саха(Якутия)": "Республика Саха (Якутия)",
    

    "Амурская область": "Амурская область",
    "Камчатский край": "Камчатский край",
    "Магаданская область": "Магаданская область",
    "Приморский край": "Приморский край",
    "Сахалинская область": "Сахалинская область",
    "Хабаровский край": "Хабаровский край",
    "Чукотский авт. округ": "Чукотский автономный округ",
    "Чукотский автономный округ": "Чукотский автономный округ",
    "Еврейская автономная область": "Еврейская автономная область",


    "г. Москва": "Москва",
    "г.Москва": "Москва",
    "г. Санкт-Петербург": "Санкт-Петербург",
    "город Санкт-Петербург": "Санкт-Петербург",
    "город Санкт - Петербург": "Санкт-Петербург",
    "Город Санкт-Петербург город федерального значения": "Санкт-Петербург",
    "Город Москва столица Российской Федерации город федерального значения": "Москва",
    "город Москва": "Москва",
    "г. Севастополь": "Севастополь",
    "город Севастополь": "Севастополь",
    
   
    "Республика Крым": "Республика Крым",
    
   
    "Российская Федерация": None,
    "г. Байконур": None,
    "Главное медицинское управление Управления делами Президента Российской Федерации": None,
    "Раздел 1. Муниципальные образования субъектов Российской Федерации": None,
    "Республика": None,
    "Поволжский район": None,
    "Приволжский": None,
    "Приволжский федеральный округ": None,
    "Центральный федеральный округ": None,
    "Центральный район": None,
    "Центральный": None,
    "Центрально-Черноземный район": None,
    "Северо-Западный федеральный округ": None,
    "Северо-Западный": None,
    "Северо-Западный район": None,
    "Северо-Кавказский федеральный округ": None,
    "Северо-Кавказский федеральный округ (до 03.06.2014)": None,
    "Северо-Кавказский федеральный округ (с 03.06.2014)": None,
    "Северо-Кавказский район": None,
    "Северо-Кавказский": None,
    "Северный район": None,
    "Уральский федеральный округ": None,
    "Уральский район": None,
    "Уральский": None,
    "Западно-Сибирский район": None,
    "Волго-Вятский район": None,
    "Восточно-Сибирский район": None,
    "Дальневосточный федеральный округ": None,
    "Дальневосточный район": None,
    "Дальневосточный": None,
    "Крымский федеральный округ": None,
    "Сибирский федеральный округ": None,
    "Сибирский": None,
    "Южный федеральный округ": None,
    "Южный федеральный округ  (до 03.06.2014)": None,
    "Южный федеральный округ  (с 03.06.2014)": None,
    "Южный федеральный округ (по 2009 год)": None,
    "Южный федеральный округ (с 2010 года)": None,
    "Южный федеральный округ (с 28.07.2016)": None,
    "Южный федеральный округ (с 29.07.2016)": None,
    "Южный": None,
    

    "автономного округа": None,
    "автономный округ": None,
    "автономный округ - Югра": None,
    "автономных округов": None,
    "Коми-Пермяцкий округ, входящий в состав Пермского края": None,
    "Корякский округ, входящий в состав Камчатского края": None,
    "Таймырский (Долгано-Ненецкий) автономный округ (Красноярский край)": None,
    "Эвенкийский автономный округ (Красноярский край)": None,
    "Bолгоградская область": "Волгоградская область",
    "Агинский Бурятский округ (Забайкальский край)": "Забайкальский край",
    "Город федерального значения Севастополь": "Севастополь",
    "Еврейская авт. область": "Еврейская автономная область",
    "Забайкальский край": "Забайкальский край",
    "Москва в старых границах": "Москва",
    "Ненецкий авт. округ": "Ненецкий автономный округ",
    "Ненецкий автономный округ": "Ненецкий автономный округ",
    "Ненецкий автономный округ (Архангельская область)": "Ненецкий автономный округ",
    "Оренбургская область": "Оренбургская область",
    "Республика Адыгея (Адыгея) (до 03.06.2014)": "Республика Адыгея",
    "Республика Карелия": "Республика Карелия",
    "Республика Коми": "Республика Коми",
    "Республика Северная": "Республика Северная Осетия - Алания",
    "Усть-Ордынский Бурятский округ": None,
    "Чеченская и Ингушская Республики": None,
    "федеральный округ": None,
    "федеральный округ3)": None,
}

# Extract all mapped values
all_values = list(region_mapping.values())

# Remove None values (invalid or aggregated entries)
valid_values = [v for v in all_values if v is not None]

# Keep only unique standardized region names
unique_standard_regions = sorted(set(valid_values))

# Display results
print("Number of reference (standardized) regions:", len(unique_standard_regions))
print(unique_standard_regions)

Number of reference (standardized) regions: 85
['Алтайский край', 'Амурская область', 'Архангельская область', 'Астраханская область', 'Белгородская область', 'Брянская область', 'Владимирская область', 'Волгоградская область', 'Вологодская область', 'Воронежская область', 'Еврейская автономная область', 'Забайкальский край', 'Ивановская область', 'Иркутская область', 'Кабардино-Балкарская Республика', 'Калининградская область', 'Калужская область', 'Камчатский край', 'Карачаево-Черкесская Республика', 'Кемеровская область', 'Кировская область', 'Костромская область', 'Краснодарский край', 'Красноярский край', 'Курганская область', 'Курская область', 'Ленинградская область', 'Липецкая область', 'Магаданская область', 'Москва', 'Московская область', 'Мурманская область', 'Ненецкий автономный округ', 'Нижегородская область', 'Новгородская область', 'Новосибирская область', 'Омская область', 'Оренбургская область', 'Орловская область', 'Пензенская область', 'Пермский край', 'Приморский кр

In [75]:
import os

def standardize_and_save(
    df,
    region_col,
    mapping_dict,
    output_folder="../../outputs/standardized_datasets",
    dataset_name="dataset"
):
    """
    Standardizes region names in a dataframe and saves the result.

    Parameters:
    - df: input dataframe
    - region_col: column containing region names
    - mapping_dict: dictionary for region name normalization
    - output_folder: folder to save the standardized dataset
    - dataset_name: output file name
    """

    print(f"\nProcessing dataset: {dataset_name}")

    # Clean region names in dataframe (remove extra spaces)
    df["_region_clean"] = df[region_col].str.strip()

    # Clean mapping keys (remove extra spaces)
    clean_mapping = {k.strip(): v for k, v in mapping_dict.items()}

    # Apply mapping
    df["region_standard"] = df["_region_clean"].map(clean_mapping)

    # Remove temporary column
    df = df.drop(columns=["_region_clean"])

    # Check unmatched regions
    unmatched = df.loc[df["region_standard"].isna(), region_col].unique()
    if len(unmatched) > 0:
        print("Unmatched regions:")
        for r in unmatched:
            print("-", r)
    else:
        print("All regions successfully mapped")

    # Create output folder if it does not exist
    os.makedirs(output_folder, exist_ok=True)

    # Save result
    output_path = os.path.join(output_folder, f"{dataset_name}.csv")
    df.to_csv(output_path, index=False)

    print(f"Saved to: {output_path}")

    return df

In [76]:
# Dictionary of datasets with corresponding region column names
datasets = {
    "child_mortality_rural_standardized": (df_child_rural, "region"),
    "child_mortality_urban_standardized": (df_child_urban, "region"),
    "disabled_standardized": (df_disabled, "region"),
    "drug_alco_standardized": (df_drug_alco, "region"),
    "gross_standardized": (df_gross, "region"),
    "housing_standardized": (df_housing, "region"),
    "morbidity_standardized": (df_morbidity, "region"),
    "newborns_standardized": (df_births_by_year, "region"),
    "population_standardized": (df_population_fixed, "region"),
    "poverty_socdem_standardized": (df_socdem_all, "region"),
    "poverty_standardized": (df_poverty, "region"),
    "real_income_standardized": (df_merged_income, "region"),
    "regional_production_standardized": (df_regional_production, "region"),
    "retail_standardized": (df_retail, "region"),
    "welfare_standardized": (df_welfare_long, "region"),
}

# Apply standardization to all datasets
for name, (df, region_col) in datasets.items():
    if region_col not in df.columns:
        print(f"Warning: dataset '{name}' does not contain column '{region_col}'. Skipped.")
        continue

    standardize_and_save(
        df=df,
        region_col=region_col,
        mapping_dict=region_mapping,
        dataset_name=name
    )


Processing dataset: child_mortality_rural_standardized
Unmatched regions:
- Российская Федерация
-     Центральный федеральный округ
-     Северо-Западный федеральный округ
-     Южный федеральный округ (по 2009 год)
-     Южный федеральный округ (с 2010 года)
-     Северо-Кавказский федеральный округ
-     Приволжский федеральный округ
-             Коми-Пермяцкий округ, входящий в состав Пермского края
-     Уральский федеральный округ
-     Сибирский федеральный округ
-             Таймырский (Долгано-Ненецкий) автономный округ (Красноярский край)
-             Эвенкийский автономный округ (Красноярский край)
-             Усть-Ордынский Бурятский округ
-     Дальневосточный федеральный округ
-             Корякский округ, входящий в состав Камчатского края
- Чеченская и Ингушская Республики
- Северный район
- Северо-Западный район
- Центральный район
- Волго-Вятский район
- Центрально-Черноземный район
- Поволжский район
- Северо-Кавказский район
- Уральский район
- Западно-Сибирс

## Conclusion

At this stage, all datasets have been fully prepared for further analysis.

The data has been:
- cleaned and transformed into a consistent format  
- standardized across datasets  
- normalized with respect to region naming  

Standardized versions of all tables have been saved.

This provides a solid foundation for merging the datasets into a unified master dataset and performing further analysis.
